# ADMET Prediction - Exploratory Data Analysis

This notebook explores the TDC ADMET datasets and molecular features.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import ADMETDataLoader, MolecularFeaturizer, get_available_datasets

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Available Datasets

In [ ]:
datasets = get_available_datasets()
pd.DataFrame(datasets).T

## 2. Load and Explore Caco-2 Dataset

In [ ]:
loader = ADMETDataLoader('Caco2_Wang')
data = loader.load('../data/raw')
data.head()

In [ ]:
# Target distribution
fig, ax = plt.subplots()
data['Y'].hist(bins=50, ax=ax)
ax.set_xlabel('log Papp (cm/s)')
ax.set_ylabel('Count')
ax.set_title('Caco-2 Permeability Distribution')
plt.tight_layout()
plt.show()

## 3. Molecular Descriptors

In [ ]:
featurizer = MolecularFeaturizer()

# Compute descriptors for all molecules
descriptors = []
for smiles in data['Drug'].head(100):
    desc = featurizer.get_molecular_descriptors(smiles)
    if desc:
        desc['smiles'] = smiles
        descriptors.append(desc)

desc_df = pd.DataFrame(descriptors)
desc_df.describe()

In [ ]:
# Correlation with target
merged = data.head(100).merge(desc_df, left_on='Drug', right_on='smiles')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
properties = ['molecular_weight', 'logp', 'tpsa', 'num_h_donors', 'num_h_acceptors', 'num_rotatable_bonds']

for ax, prop in zip(axes.flat, properties):
    ax.scatter(merged[prop], merged['Y'], alpha=0.5)
    ax.set_xlabel(prop)
    ax.set_ylabel('log Papp')

plt.tight_layout()
plt.show()

## 4. Graph Statistics

In [ ]:
# Featurize sample molecules
sample_smiles = data['Drug'].head(10).values
sample_labels = data['Y'].head(10).values

graphs = featurizer.featurize_dataset(sample_smiles, sample_labels, show_progress=False)

for i, g in enumerate(graphs):
    print(f"Molecule {i+1}: {g.x.shape[0]} atoms, {g.edge_index.shape[1]//2} bonds")

In [ ]:
print(f"\nAtom feature dimension: {featurizer.atom_feature_dim}")
print(f"Bond feature dimension: {featurizer.bond_feature_dim}")